# Decision Tree - Cross-Validation and Dataset Comparison

This notebook evaluates the custom Decision Tree implementation (`models/decision_tree.py`) using k-fold cross-validation.

Objectives:
1. Compare KFold (no stratification) vs StratifiedKFold on engineered and original datasets.
2. Compare performance on `results/fires_engineered_features.csv` (engineered) vs `results/fires_merged_all_features.csv` (original).
3. Report metrics: accuracy, precision, recall, F1-score; also inspect tree depth and number of leaves per fold.

In [1]:
# Imports and setup
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

# Add models path and import the custom DecisionTree
sys.path.insert(0, os.path.abspath('../'))
from models.decision_tree import DecisionTree

# Plot style
sns.set_palette('husl')
plt.style.use('seaborn-v0_8-darkgrid')

print('✅ Imports and custom DecisionTree loaded')

✅ Imports and custom DecisionTree loaded


In [2]:
# Load datasets
df_eng = pd.read_csv('../results/fires_engineered_features.csv')
df_orig = pd.read_csv('../results/fires_merged_all_features.csv')

print('Engineered dataset:', df_eng.shape)
print(df_eng['class'].value_counts())
print('Original dataset:', df_orig.shape)
print(df_orig['class'].value_counts())

Engineered dataset: (7302, 23)
class
0    4017
1    3285
Name: count, dtype: int64
Original dataset: (7302, 43)
class
0    4017
1    3285
Name: count, dtype: int64


## Prepare datasets for modeling

We keep `longitude`, `latitude` and `class` out of features, ensure numeric-only features for the original dataset, and fill missing values if present. Decision trees do not require scaling (scaling is not applied here).

In [3]:
# ENGINEERED dataset
X_eng = df_eng.drop(['longitude', 'latitude', 'class'], axis=1)
y_eng = df_eng['class']

# ORIGINAL dataset - numeric features only
X_orig = df_orig.drop(['longitude', 'latitude', 'class'], axis=1)
# If there are non-numeric columns, keep only numeric columns
non_numeric_cols = X_orig.select_dtypes(include=['object']).columns.tolist()
if non_numeric_cols:
    print(f'⚠️ Dropping non-numeric columns from original dataset: {non_numeric_cols}')
    X_orig = X_orig.select_dtypes(include=[np.number])

# Fill missing values with column means if needed
if X_orig.isnull().sum().sum() > 0:
    print('⚠️ Missing values detected in original dataset, filling with column means')
    X_orig = X_orig.fillna(X_orig.mean())

y_orig = df_orig['class']

print('Prepared shapes:')
print('X_eng:', X_eng.shape, 'y_eng:', y_eng.shape)
print('X_orig:', X_orig.shape, 'y_orig:', y_orig.shape)

Prepared shapes:
X_eng: (7302, 20) y_eng: (7302,)
X_orig: (7302, 40) y_orig: (7302,)


## Cross-Validation Evaluation Function (custom DecisionTree)

We implement a helper to evaluate the custom DecisionTree over each fold and return fold-wise metrics, tree depth and leaf counts. We'll also gather concatenated predictions and true labels for an overall confusion matrix.

In [4]:
def evaluate_cv_custom_tree(X, y, cv, *, max_depth=None, min_samples_split=2, min_samples_leaf=1, random_state=42):
    X = np.array(X)
    y = np.array(y)

    fold_metrics = []
    depths = []
    n_leaves = []
    all_y_true = []
    all_y_pred = []

    for fold_idx, (train_idx, test_idx) in enumerate(cv.split(X, y), start=1):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        model = DecisionTree(max_depth=max_depth,
                             min_samples_split=min_samples_split,
                             min_samples_leaf=min_samples_leaf)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, zero_division=0)
        rec = recall_score(y_test, y_pred, zero_division=0)
        f1 = f1_score(y_test, y_pred, zero_division=0)

        depth = model.get_depth()
        leaves = model.get_n_leaves()

        fold_metrics.append({
            'fold': fold_idx,
            'accuracy': acc,
            'precision': prec,
            'recall': rec,
            'f1_score': f1,
            'depth': depth,
            'n_leaves': leaves
        })

        depths.append(depth)
        n_leaves.append(leaves)

        all_y_true.extend(y_test.tolist())
        all_y_pred.extend(y_pred.tolist())

    metrics_df = pd.DataFrame(fold_metrics)
    cm = confusion_matrix(all_y_true, all_y_pred)

    return metrics_df, cm

print('✅ Evaluation function ready')

✅ Evaluation function ready


## Run evaluations: engineered vs original datasets, KFold vs StratifiedKFold

We'll run 5-fold CV for both KFold (no stratification) and StratifiedKFold for each dataset.

In [5]:
n_splits = 2
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

# Engineered dataset (already numeric)
metrics_eng_kf, cm_eng_kf = evaluate_cv_custom_tree(X_eng, y_eng, kf)
metrics_eng_skf, cm_eng_skf = evaluate_cv_custom_tree(X_eng, y_eng, skf)

# Original dataset (after numeric selection)
metrics_orig_kf, cm_orig_kf = evaluate_cv_custom_tree(X_orig, y_orig, kf)
metrics_orig_skf, cm_orig_skf = evaluate_cv_custom_tree(X_orig, y_orig, skf)

print('✅ CV runs complete for all combinations')

KeyboardInterrupt: 

## Display fold-wise and aggregated metrics

We show per-fold results and compute mean ± std for each combination.

In [ ]:
def summarize_metrics(metrics_df):
    summary = metrics_df[['accuracy','precision','recall','f1_score','depth','n_leaves']].agg(['mean','std']).T
    summary.columns = ['mean','std']
    return summary

print('Engineered - KFold:')
display(metrics_eng_kf)
display(summarize_metrics(metrics_eng_kf))

print('Engineered - StratifiedKFold:')
display(metrics_eng_skf)
display(summarize_metrics(metrics_eng_skf))

print('Original - KFold:')
display(metrics_orig_kf)
display(summarize_metrics(metrics_orig_kf))

print('Original - StratifiedKFold:')
display(metrics_orig_skf)
display(summarize_metrics(metrics_orig_skf))

## Visual Comparisons

Plot aggregated mean F1-score (and std) across the four combinations to compare the effect of stratification and feature engineering.

In [ ]:
# Prepare summary metrics for plotting
def mean_std_row(metrics_df):
    return metrics_df[['accuracy','precision','recall','f1_score','depth','n_leaves']].mean(), metrics_df[['accuracy','precision','recall','f1_score','depth','n_leaves']].std()

eng_kf_mean, eng_kf_std = mean_std_row(metrics_eng_kf)
eng_skf_mean, eng_skf_std = mean_std_row(metrics_eng_skf)
orig_kf_mean, orig_kf_std = mean_std_row(metrics_orig_kf)
orig_skf_mean, orig_skf_std = mean_std_row(metrics_orig_skf)

summary_df = pd.DataFrame({
    'Engineered_KF': eng_kf_mean,
    'Engineered_SKF': eng_skf_mean,
    'Original_KF': orig_kf_mean,
    'Original_SKF': orig_skf_mean
}).T

summary_std_df = pd.DataFrame({
    'Engineered_KF': eng_kf_std,
    'Engineered_SKF': eng_skf_std,
    'Original_KF': orig_kf_std,
    'Original_SKF': orig_skf_std
}).T

# Plot F1-score bar chart with error bars
fig, ax = plt.subplots(figsize=(10, 6))
f1_means = summary_df['f1_score']
f1_err = summary_std_df['f1_score']
ax.bar(f1_means.index, f1_means.values, yerr=f1_err.values, capsize=8, alpha=0.85)
ax.set_ylabel('F1-Score (mean across folds)')
ax.set_title('Custom Decision Tree: F1 Score Comparison (Engineered vs Original; KFold vs StratifiedKFold)')
ax.set_ylim(0, 1.05)
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

# Plot accuracy, precision, recall similarly
metrics_to_plot = ['accuracy','precision','recall']
fig, axes = plt.subplots(1, len(metrics_to_plot), figsize=(18, 5))
for i, metric in enumerate(metrics_to_plot):
    means = summary_df[metric]
    errs = summary_std_df[metric]
    axes[i].bar(means.index, means.values, yerr=errs.values, capsize=8, alpha=0.85)
    axes[i].set_title(metric.capitalize())
    axes[i].set_ylim(0, 1.05)
    axes[i].tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

## Confusion Matrices (aggregated across folds)

We plot the concatenated confusion matrix for each combination.

In [ ]:
cms = {
    'Engineered_KF': cm_eng_kf,
    'Engineered_SKF': cm_eng_skf,
    'Original_KF': cm_orig_kf,
    'Original_SKF': cm_orig_skf
}

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.ravel()
for ax, (label, cm) in zip(axes, cms.items()):
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, cbar=False)
    ax.set_title(label.replace('_',' '))
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.set_xticklabels(['No Fire (0)', 'Fire (1)'])
    ax.set_yticklabels(['No Fire (0)', 'Fire (1)'])

plt.tight_layout()
plt.show()

## Final Summary & Conclusions

Summarize findings on whether stratification mattered for the Decision Tree performance and whether engineered features improved performance relative to original features.

In [ ]:
def print_summary():
    print('---- Engineered dataset ----')
    print('KFold mean F1:', summarize_metrics(metrics_eng_kf).loc['f1_score','mean'])
    print('StratifiedKFold mean F1:', summarize_metrics(metrics_eng_skf).loc['f1_score','mean'])
    print('---- Original dataset ----')
    print('KFold mean F1:', summarize_metrics(metrics_orig_kf).loc['f1_score','mean'])
    print('StratifiedKFold mean F1:', summarize_metrics(metrics_orig_skf).loc['f1_score','mean'])
    
print_summary()